In [1]:
# Computes stochastic closure certificate using SOS for stochastic systems in verifying LTL specifications
# Random Walk Example

# include important libraries
using JuMP
using MosekTools
using DynamicPolynomials
using MultivariatePolynomials
using LinearAlgebra
using TSSOS # important for SOS, see https://github.com/wangjie212/TSSOS
using Distributions # for the noise

@polyvar x y x0 # global vars used in monomials
var = [x, y]
sos_tol = 1 # the maximum degree of unknown SOS lagrange polynomials = deg + sos_tol 
error = 2   # precision digit places
gamma, lamda, delta, tau1, tau2 = 0.03, 500, 15, 0.001, 0.001 # sat prob and S-procedure parameters

# uniform noise generator ~ U([-2, 1])
function noise()
    return rand(Uniform(-2, 1))
end

# access odd priorities in DPA
function odd_pri(d::Dict)
    return [v for v in values(d) if isodd(v)]
end

# creates the parametrized SCC
function add_paramC_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="c_$(q)_$(p)")
    C_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return C_qp, coeffs, basis
end

function add_paramCf_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="cf_$(q)_$(p)")
    Cf_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return Cf_qp, coeffs, basis
end

# creates the parametrized ranking functions
function add_paramV_poly!(model, var2, deg, l, q, p)
    basis = monomials(var2, 0:deg)
    coeffs = @variable(model, [1:length(basis)], base_name="v_$(l)_$(q)_$(p)")
    V_l_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return V_l_qp, coeffs, basis
end

# the domain is function pairwise
# X0=[2,3]
ϵ = 1e-6
bd = 150 
gXplus, gXYplus = [x-100-ϵ, bd-x], [x+bd, bd-x, y-100-ϵ, bd-y] # domain is (100,bd]
gXminus, gXYminus = [x+bd, 100-x], [x+bd, bd-x, y+bd, 100-y] # domain is [-bd,100]
gXY = [x+bd, bd-x, y+bd, bd-y] # entire domain
g5 = [x0-2, 3-x0, x+bd, bd-x, y+bd, bd-y]

# polynomial stochastic closure certificate of degree deg
function scc(deg, S, pri)
    # synthesize SCC by using the standard formulation
    # deg: degree of scc template
    model = Model(optimizer_with_attributes(Mosek.Optimizer))
    set_optimizer_attribute(model, MOI.Silent(), true)
    
    C_dict = Dict()  # key => (symbolic_poly, numeric_poly)
    for q in keys(S1) # DPA states
        for (lab, qn) in S[q]
            C, Cc, Cb = add_paramC_poly!(model, var, deg, q, qn) # CC, CC coefficients, and CC basis functions
            Cf, Ccf, Cbf = add_paramCf_poly!(model, var, deg, q, qn)
            key_c = (q, qn, lab, "c") 
            key_cf = (q, qn, lab, "cf")
            C_dict[key_c] = (C, Cc, Cb)  # store symbolic, coefficients, basis
            C_dict[key_cf] = (Cf, Ccf, Cbf)

            # cond 1
            C_plus1 = C([x, y]=>[x, x]) # over x>100
            C_minus1 = C([x, y]=>[x, x + noise()]) # over x<=100
            E_C1plus, E_C1minus = C_plus1, C_minus1 - 0.5
            add_psatz!(model, -E_C1plus + gamma, [x], gXplus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
            add_psatz!(model, -E_C1minus + gamma, [x], gXminus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)

            # cond 2
            if pri[q] <= pri[qn]
                C_plus2 = Cf([x, y]=>[x, x])
                C_minus2 = Cf([x, y]=>[x, x + noise()])
                E_Cf2plus, E_Cf2minus = C_plus2, C_minus2 - 0.5
                add_psatz!(model, -E_Cf2plus + gamma, [x], gXplus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                add_psatz!(model, -E_Cf2minus + gamma, [x], gXminus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
            end

            for p in keys(S1)
                # cond 3
                Cf3e, Ccf3e, Cbf3e = add_paramCf_poly!(model, var, deg, p, qn) # e as in E[.\.]
                Cf3, Ccf3, Cbf3 = add_paramCf_poly!(model, var, deg, p, q) 
                key_cf3 = (p, q, "cf") 
                key_cf3e = (p, qn, "cf")
                C_dict[key_cf3] = (Cf3, Ccf3, Cbf3)  
                C_dict[key_cf3e] = (Cf3e, Ccf3e, Cbf3e)
                if pri[p] <= pri[q] && pri[p] <= pri[qn] 
                    Cf_plus3 = Cf3e([x, y]=>[x, y])
                    Cf_minus3 = Cf3e([x, y]=>[x, y + noise()])
                    E_Cf3plus, E_Cf3minus = Cf_plus3, Cf_minus3 - 0.5
                    add_psatz!(model, Cf3 - E_Cf3plus, [x, y], gXYplus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)  
                    add_psatz!(model, Cf3 - E_Cf3minus, [x, y], gXYminus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)              
                end

                # cond 4
                C4e, Cc4e, Cb4e = add_paramC_poly!(model, var, deg, p, qn) # e as in E[.\.]
                C4, Cc4, Cb4 = add_paramC_poly!(model, var, deg, p, q) 
                key_c4 = (p, q, "c") 
                key_c4e = (p, qn, "c")
                C_dict[key_c4] = (C4, Cc4, Cb4)  
                C_dict[key_c4e] = (C4e, Cc4e, Cb4e)
                C_plus4 = C4e([x, y]=>[x, y])
                C_minus4 = C4e([x, y]=>[x, y + noise()])
                E_C4plus, E_C4minus = C_plus4, C_minus4 - 0.5
                add_psatz!(model, C4 - E_C4plus, [x, y], gXYplus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                add_psatz!(model, C4 - E_C4minus, [x, y], gXYminus, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)

                #### non-negativity of C and Cf
                add_psatz!(model, C4, var, gXY, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
                add_psatz!(model, Cf3, var, gXY, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                
                # cond 5
                for l in odd_pri(pri)
                    V2, Vc2, Vb2 = add_paramV_poly!(model, [x0, y], deg, l, "q0", q)
                    V1, Vc1, Vb1 = add_paramV_poly!(model, [x0, x], deg, l, "q0", p)
                    key_v1 = (l, "q0", p, "v")
                    key_v2 = (l, "q0", q, "v")
                    C_dict[key_v1], C_dict[key_v2] = (V1, Vc1, Vb1), (V2, Vc2, Vb2)
                    #### non-negativity of V
                    add_psatz!(model, V1, var, gXY, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                    add_psatz!(model, V2, var, gXY, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
            
                    C1 = C4([x, y]=>[x0, x])
                    if l <= pri[p] && pri[p] <= pri[q]
                        add_psatz!(model, -V2 + V1 - delta + tau1*(C1 - lamda) + tau2*(Cf3 - lamda), [x0,x,y], g5, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                    end
                end
            end
        end
    end
        
    optimize!(model) # solve for coefficients
    status = termination_status(model)
    C_eval_dict = Dict() # Get numerical values of coefficients and plug into polynomials
    for (key, (C, Cc, Cb)) in C_dict
        coeff_vals = round.(value.(Cc); digits=error)  # Round each coefficient to 2 decimal places
        C_numeric = sum(coeff_vals[i] * Cb[i] for i in eachindex(Cb))
        C_eval_dict[key] = (C, C_numeric)
    end
    
    # status might be optimal but if all Bc approx 10^{-error}, it's essentially 0 so OPTIMAL != CC.
    p_sat = 1-gamma/lamda # satisfaction probability lower bound
    return status, C_eval_dict, p_sat
end

# the considered DPA 

# A representing Gb
S1 = Dict(
    :"q0" => [("a", :"q0"), ("b", :"q0"), ("c", :"q1")],
    :"q1" => [("a", :"q1"), ("b", :"q1"), ("c", :"q1")]
)
pri1 = Dict(:"q0" => 2, :"q1" => 1) # priority function

# A representing GFa
S2 = Dict(
    :"q0" => [("b", :"q0"), ("a", :"q1"), ("c", :"q0")],
    :"q1" => [("a", :"q1"), ("b", :"q0"), ("c", :"q0")]
)
pri2 = Dict(:"q0" => 1, :"q1" => 2) # priority function

# A representing bUa
S3 = Dict(
    :"q0" => [("b", :"q0"), ("a", :"q1"), ("c", :"q2")],
    :"q1" => [("a", :"q1"), ("b", :"q1"), ("c", :"q1")],
    :"q2" => [("a", :"q2"), ("b", :"q2"), ("c", :"q2")]
)
pri3 = Dict(:"q0" => 3, :"q1" => 2, :"q2" => 3)

# collection  of DPA
S_list = Dict(1=>S1, 2=>S2, 3=>S3)
pri_list = Dict(1=>pri1, 2=>pri2, 3=>pri3)

# Simulation
deg = 1 # desired degree of SCC
for k in 1:length(S_list)
    stats = @timed (status, CC_data, p_sat) = scc(deg, S_list[k], pri_list[k])
    
    println("poly deg: ", deg)
    println("status: ", status)
    println("sat probability: ", p_sat)
    println("Number of SCC polynomials: ", length(CC_data))
    println("time: ", stats.time, "\n")

    # Write the dictionary line-by-line
    for (kk, v) in CC_data
        println(kk, ": ", v[2], ",")
    end
end

println("Finished")

poly deg: 1
status: OPTIMAL
sat probability: 0.99994
Number of SCC polynomials: 22
time: 10.999859542

("q1", "q1", "c", "cf"): -325.48 + 202.64*y - 204.46*x,
("q0", "q1", "c", "c"): -443.43 - 62.06*y + 61.37*x,
("q0", "q1", "cf"): 0.0,
("q0", "q1", "c"): -386.37 - 1.86*y,
("q1", "q1", "a", "cf"): -430.38 + 96.65*y - 97.44*x,
("q1", "q1", "c", "c"): -412.73 + 127.54*y - 128.48*x,
(1, "q0", "q0", "v"): 100.63 + 407.52*x0,
("q0", "q0", "b", "c"): -452.17 - 0.31*y - 0.32*x,
("q1", "q0", "c"): 757.12 - 0.08*y,
("q0", "q0", "a", "c"): -318.35 + 205.96*y - 207.87*x,
("q1", "q1", "a", "c"): -448.58 - 39.96*y + 39.3*x,
("q1", "q1", "b", "cf"): -449.28 + 35.89*y - 36.53*x,
("q1", "q1", "cf"): -386.39 - 1.86*y,
(1, "q0", "q1", "v"): 718.23 + 411.4*x0,
("q0", "q0", "b", "cf"): -448.06 - 42.74*y + 42.08*x,
("q0", "q0", "c"): 757.38 - 0.1*y,
("q1", "q1", "b", "c"): -407.1 - 135.97*y + 134.97*x,
("q1", "q0", "cf"): 757.42 - 0.1*y,
("q0", "q0", "a", "cf"): -423.59 - 110.17*y + 109.32*x,
("q1", "q1", 

#### To post verify the SOS SC2 via Z3 and remember to change the kernel to python

In [79]:
# SC^2 for Gambler's ruin without DPA

from z3 import *
import itertools
import numpy as np
import time
from itertools import product

# --- define symbolic polynomial variables ---
x0, x, y = Reals('x0 x y')
# we check each candidate iteratively
############################ Dynamics ###############
# SC2 from SOS
D = {
# ("q1", "q1", "c", "cf"): -314.81 + 56.95*y - 57.46*x,
("q0", "q1", "cf"): 0.0,
# ("q0", "q1", "c"): -278.7 - 1.23*y,
# ("q1", "q1", "a", "cf"): -290.05 - 101.97*y + 101.26*x,
# ("q1", "q1", "c", "c"): -290.13 + 101.54*y - 102.25*x,
("q1", "q2", "c"): -278.8 - 1.26*y,
# ("q0", "q2", "c", "cf"): -260.33 + 131.1*y - 132.1*x,
# ("q1", "q2", "cf"): -278.8 - 1.26*y,
# ("q0", "q0", "b", "c"): -320.76 + 36.97*y - 37.42*x,
("q1", "q0", "c"): 546.21 - 0.06*y,
("q0", "q2", "c"): -278.69 - 1.25*y,
(3, "q0", "q0", "v"): 70.78 + 294.12*x0,
# ("q0", "q2", "c", "c"): -324.38 + 13.9*y - 14.33*x,
# ("q1", "q1", "a", "c"): -240.71 + 143.87*y - 145.08*x,
# ("q0", "q1", "a", "c"): -298.32 + 90.0*y - 90.63*x,
# ("q1", "q1", "b", "cf"): -324.92 - 3.62*y + 3.2*x,
# ("q0", "q2", "cf"): -278.69 - 1.25*y,
# ("q1", "q1", "cf"): -278.83 - 1.27*y,
("q0", "q1", "a", "cf"): 0.0,
# ("q0", "q0", "b", "cf"): -265.89 + 126.68*y - 127.62*x,
("q0", "q0", "c"): 546.53 - 0.06*y,
# ("q1", "q1", "b", "c"): -249.26 + 138.8*y - 139.91*x,
("q1", "q0", "cf"): 546.21 - 0.06*y,
(3, "q0", "q1", "v"): 188.76 + 188.76*x0,
("q1", "q1", "c"): -278.8 - 1.23*y,
("q0", "q0", "cf"): 546.53 - 0.06*y
}

# --- parameters ---
gamma, lamda, delta, tau1, tau2 = 0.03, 500, 15, 0.001, 0.001

def noise():
    return np.random.uniform(-2, 1)

bd = 150

def f(x):
    xn = x + noise()
    xn_clipped = If(Or(xn<-bd, xn>bd), x, xn)
    return If(x > 100, x, xn_clipped)

# # A representing Gb
# S1 = {
#     "q0": [("a", "q0"), ("b", "q0"), ("c", "q1")],
#     "q1": [("a", "q1"), ("b", "q1"), ("c", "q1")]
# }
# pri = {"q0": 2, "q1": 1} # priority function

# # A representing GFa
# S1 = {
#     "q0": [("b", "q0"), ("a", "q1"), ("c", "q0")],
#     "q1": [("a", "q1"), ("b", "q0"), ("c", "q0")]
# }
# pri = {"q0": 1, "q1": 2} # priority function

# A representing bUa
S1 = {
    "q0": [("b", "q0"), ("a", "q1"), ("c", "q2")],
    "q1": [("a", "q1"), ("b", "q1"), ("c", "q1")],
    "q2": [("a", "q2"), ("b", "q2"), ("c", "q2")]
}
pri = {"q0": 3, "q1": 2, "q2": 3}
########################################################

odd_pri = [v for v in pri.values() if v % 2 == 1] # odd priorities

def to_z3(e): # convert SCCs to z3 liftable expressions
    return e if is_expr(e) else RealVal(e)

# --- Constraints on domains ---
s = Solver()

for q in S1.keys():
    for (lab, qn) in S1[q]:
        # cond 1
        key1 = (q,qn,lab,"c")
        if key1 in D:
            C1 = to_z3(D[key1])
            E_C1 = substitute(C1, (y, f(x)))
            s.add(ForAll(x, Implies(x > 100, E_C1 <= gamma)))
            s.add(ForAll(x, Implies(x <= 100, E_C1 - RealVal(0.5) <= gamma)))

        # cond 2
        key2 = (q,qn,lab,"cf")
        if key2 in D:
            Cf2 = to_z3(D[key2])
            E_Cf2 = substitute(Cf2, (y, f(x)))
            if pri[q] <= pri[qn]:
                s.add(ForAll(x, Implies(x > 100, E_Cf2 <= gamma)))
                s.add(ForAll(x, Implies(x <= 100, E_Cf2 - RealVal(0.5) <= gamma)))

        for p in S1.keys():
            # cond 3
            key3, key3e = (p,q,"cf"), (p,qn,"cf")
            if key3 not in D or key3e not in D:
                continue   # skip this iteration
            Cf3, Cf3e = to_z3(D[key3]), to_z3(D[key3e]) ######
            s.add(ForAll([x, y], Implies(And(-bd<=x, x<=bd, -bd<=y, y<=bd), Cf3 >= 0)))
            if pri[p] <= pri[q] and pri[p] <= pri[qn]:
                E_Cf3 = substitute(Cf3e, (y, f(y)))
                s.add(ForAll([x, y], Implies(And(-bd<=x, x<=bd, y>100, y<=bd), E_Cf3 <= Cf3)))
                s.add(ForAll([x, y], Implies(And(-bd<=x, x<=bd, y<=100, -bd<=y), E_Cf3 - RealVal(0.5) <= Cf3)))

            # cond 4
            key4, key4e = (p,q,"c"), (p,qn,"c")
            if key4 not in D or key4e not in D:
                continue   # skip this iteration
            C4, C4e = to_z3(D[key4]), to_z3(D[key4e]) ########
            s.add(ForAll([x, y], Implies(And(-bd<=x, x<=bd, -bd<=y, y<=bd), C4 >= 0)))
            E_C4 = substitute(C4e, (y, f(y)))
            s.add(ForAll([x, y], Implies(And(-bd<=x, x>=bd, y>100, y<=bd), E_C4 <= C4)))
            s.add(ForAll([x, y], Implies(And(-bd<=x, x>=bd, y<=100, -bd<=y), E_C4 - RealVal(0.5) <= C4)))

            # cond 5
            for l in odd_pri:
                key5a, key5b = (l,"q0",p,"v"), (l,"q0",q,"v")
                if key5a in D and key5b in D:
                    V5a = D[key5a]
                    V5b = D[key5b]
                    if l <= pri[p] and pri[p] <= pri[q]:
                        # pass
                        s.add(ForAll([x0, x, y], Implies(And(2<=x0, x0<=3, -bd<=x, x<=bd, -bd<=y, y<=bd), V5a >= 0)))
                        s.add(ForAll([x0, x, y], Implies(And(2<=x0, x0<=3, -bd<=x, x<=bd, -bd<=y, y<=bd), V5b >= 0)))
                        s.add(V5b <= V5a - delta + tau1*(C4 - lamda) + tau2*(Cf3 - lamda))

start_solve = time.time()
result = s.check()
print("sat" if result == sat else "unsat")
end_solve = time.time()
print("Solve time:", end_solve - start_solve)

sat
Solve time: 0.013351202011108398
